In [36]:
using Random, Distributions, DataFrames, LinearAlgebra, Optim, GLM, Expectations
# using Pkg
# Pkg.add("BenchmarkTools")
Random.seed!(73723462)

#First step: Monte Carlo simulation to create our dataset

# Set up model primitives
N = 1000
theta_true = [0.5, 1, -1, 2, 0.6, 1]
beta = theta_true[1:2]
pi = theta_true[3:4]
rho = theta_true[5]
sigma = theta_true[6]

# Draw the shocks
Σ = [1.0 rho; rho sigma]           # Covariance matrix
mvnorm = MvNormal([0.0, 0.0], Σ)   # Bivariate normal with mean 0 and covariance Σ
e = rand(mvnorm, N)                # 2×N array, each column is a draw
e1 = e[1, :]
e2 = e[2, :]

# Draw Z ~ Uniform(-1,1) and construct x
z = rand(Uniform(-1,1), N)
x = pi[1] .+ pi[2] .* z .+ e2

# Construct ylat and y
ylat = beta[1] .+ beta[2] .* x .+ e1
y = Int.(ylat .> 0)

# Collect into a df - probably more computationally efficient to use a Matrix, but this will help avoid mistakes
df = DataFrame(z = z,x = x, y = y)

Row,z,x,y
,Float64,Float64,Int64
1,0.844838,2.38644,1
2,-0.0747308,-0.849774,0
3,-0.24472,0.0945088,1
4,-0.858939,-4.56404,0
5,-0.8637,-2.37241,0
6,-0.369697,-1.96426,0
7,0.633558,0.411548,1
8,-0.656636,-1.45348,1
9,-0.415549,-1.80381,0


In [37]:
# Before we code up our estimators, let's just confirm that regular probit gets the wrong answer. Using a canned routine:

probit = glm(@formula(y ~ x), df, Binomial(), ProbitLink())

StatsModels.TableRegressionModel{GeneralizedLinearModel{GLM.GlmResp{Vector{Float64}, Binomial{Float64}, ProbitLink}, GLM.DensePredChol{Float64, CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

y ~ 1 + x

Coefficients:
────────────────────────────────────────────────────────────────────────
                Coef.  Std. Error      z  Pr(>|z|)  Lower 95%  Upper 95%
────────────────────────────────────────────────────────────────────────
(Intercept)  0.749012   0.0786874   9.52    <1e-20   0.594788   0.903237
x            1.34276    0.0790714  16.98    <1e-63   1.18778    1.49773
────────────────────────────────────────────────────────────────────────

In [38]:
# Next, we will write code that sets up our MLE estimator

# Set up the likelihood function for a single observation as a function of theta
function likelihood(observation, theta)
    # Rewrite the theta vector using the names you've given the parameters - bad for RAM, but avoids mistakes!
    beta_c, beta_x, pi_c, pi_z, rho, sigma = theta 
    # could also write beta/pi here as vectors, then redefine x/z_new = [ones(N), x/z] - this problem is simple enough that I won't bother

    # Define a mapping between your observation - same comment as above (but especially helpful if you use a Matrix instead of a DataFrame)
    z_i, x_i, y_i = observation."z"[1], observation."x"[1], observation."y"[1]

    # Build the likelihood
    stage1_i = pdf(Normal(), (x_i - pi_c - pi_z * z_i) / sigma) / sigma # stage 1 likelihood (standard linear model)
    M_i = (beta_c + beta_x * x_i + (rho / sigma) * (x_i - pi_c - pi_z * z_i)) / sqrt(1 - (rho^2)) # building block for stage 2 likelihood
    
    # Use an if statement to make easier to read - don't do this if you want to use an autodiff
    if (y_i) == 1
        return cdf(Normal(), M_i) * stage1_i
    else
        return (1 - cdf(Normal(), M_i)) * stage1_i
    end
end

# Set up the MLE objective function, which simply applied likelihood() to each obs in the data, logs, and sums - note the -1!
# Also, not a problem here, but beware log(really small number) issues - floating point precision errors can arise.
# Can arise with MLE with R^2 \approx 0, or certain transformations that happen within structural models (e.g. log(market share) for tiny firm)
function mle_objective(theta; data = df)
    return -1 * sum(log.(likelihood(data[i,:], theta) for i in 1:size(df)[1]))
end

mle_objective (generic function with 1 method)

In [39]:
# Let's optimize this objective function!

# Supply an initial guess
theta_init = [0.4, 1.1, -0.1, 2.5, 0.5, 0.9]

# Define bounds on
lower = [-Inf, -Inf, -Inf, -Inf, 0.0, 0.0]  # rho >= 0, sigma >= 0
upper = [ Inf,  Inf,  Inf,  Inf, 1.0, Inf]  # rho <= 1

# Minimize using Nelder-Mead (a "derivative-free" optimzer)
result = optimize(theta -> mle_objective(theta; data = df),
                  lower, upper,
                  theta_init,    # starting point
                  NelderMead())  # solver
# Alternative syntax:
                #   Fminbox(NelderMead()))  # solver
# Note that we need to wrap Nelder-Mead in another solver to support box constraints. Another approach would be to reparameterize rho using a sigmoid and sigma by squaring
# Derivative-dependent optimzers also exist, and are usually much better. BFGS() is a simple one (still need to wrap for bounds), but beware numerical derivatives!
# This problem is nice and convex, so anything will pretty much work here. When we get to BLP, this will distinctly not be true.

theta_MLE = Optim.minimizer(result)
hcat(theta_MLE, theta_true)

6×2 Matrix{Float64}:
  0.429869   0.5
  0.993969   1.0
 -1.04197   -1.0
  1.94199    2.0
  0.585083   0.6
  1.02932    1.0

In [40]:
# The code above loops through observations in the data and individually evaluates the likelihood. 
# You can speed up your code a lot by vectorizing and broadcasting. Here's an example:

function mle_objective_vec(theta; data = df)
    z, x, y = data.z, data.x, data.y
    beta_c, beta_x, pi_c, pi_z, rho, sigma = theta

    stage1 = pdf.(Normal(), (x .- pi_c .- pi_z .* z) ./ sigma) ./ sigma
    M = (beta_c .+ beta_x .* x .+ (rho / sigma) .* (x .- pi_c .- pi_z .* z)) ./ sqrt(1 - (rho^2))
    stage2 = ifelse.(y .== 1, cdf.(Normal(), M), 1 .- cdf.(Normal(), M))
    return -sum(log.(stage2 .* stage1))
end

result_fast = optimize(theta -> mle_objective_vec(theta; data = df),
                  lower, upper,
                  theta_init,
                  Fminbox(NelderMead()))

# ...and the results are identical, as expected.

theta_MLE_fast = Optim.minimizer(result_fast)
hcat(theta_MLE_fast, theta_MLE, theta_true)

6×3 Matrix{Float64}:
  0.429862   0.429869   0.5
  0.993964   0.993969   1.0
 -1.04197   -1.04197   -1.0
  1.94199    1.94199    2.0
  0.585085   0.585083   0.6
  1.02932    1.02932    1.0

In [41]:
using BenchmarkTools

# Warm-up once to compile
mle_objective(theta_init; data=df)
mle_objective_vec(theta_init; data=df)

# Benchmark
@btime mle_objective($theta_init; data=$df)
@btime mle_objective_vec($theta_init; data=$df)

# So this simple change made my code 12x faster (on my computer) - not the only runtime optimization, but an effective one

  610.625 μs (34107 allocations: 549.14 KiB)
  37.542 μs (53 allocations: 33.52 KiB)


2394.9849140022384

In [42]:
# We can also code up the standard errors - somewhat gnarly, so let's not go into much detail 

function ivprobit_obs_llgh(theta, y, x, z)
    # Unpack parameters
    beta_c, beta_x, pi_c, pi_z, rho, sigma = theta

    # Derived quantities
    v = x - pi_c - pi_z*z
    u = v / sigma
    c = sqrt(1 - rho^2)
    n = beta_c + beta_x*x + (rho/sigma)*v
    m = n / c
    # and distributions
    Phi_m = cdf(Normal(), m)
    phi_m = pdf(Normal(), m)
    phi_u = pdf(Normal(), u)
    # where logL = y*log(Phi_m) + (1 - y)*log(1 - Phi_m) + log(phi_u) - log(sigma) 
    
    # for stability
    Phi_m = clamp(Phi_m, 1e-10, 1 - 1e-10)

    # Score factor for probit part 
    # Defined such that grad(logL) = g(m)grad(m) - ugrad(u) - 1/sigma * e_6 
    # and hess(logL) = g'(m)grad(m)grad(m)' + g(m)hess(m) - grad(u)grad(u)' - uhess(u) + 1/sigma^2 e_6e_6'
    g = phi_m * (y/Phi_m - (1 - y)/(1 - Phi_m))     # g(m)
    gp = -m*g - phi_m^2*(y/Phi_m^2 + (1 - y)/(1 - Phi_m)^2)  # g'(m)

    # First derivatives of m and u
    dm = zeros(6)
    dm[1] = 1/c
    dm[2] = x/c
    dm[3] = -rho/(sigma*c)
    dm[4] = -rho*z/(sigma*c)
    dm[5] = v/(sigma*c) + n*rho/(c^3)
    dm[6] = -rho*v/(sigma^2*c)

    du = zeros(6)
    du[3] = -1/sigma
    du[4] = -z/sigma
    du[6] = -u/sigma

    # Gradient (vector of length 6)
    grad = g*dm - u*du - [0,0,0,0,0,1/sigma]

    # Second derivatives of m and u (mostly sparse)
    d2m = zeros(6,6)
    d2u = zeros(6,6)

    # Nonzero entries of m-Hessian
    d2m[5,1] = d2m[1,5] = rho/(c^3)                # m_{r,b0}
    d2m[5,2] = d2m[2,5] = x*rho/(c^3)              # m_{r,b1}
    d2m[5,3] = d2m[3,5] = -(1/(sigma*c)) + rho^2/(sigma*c^3)
    d2m[5,4] = d2m[4,5] = -(z/(sigma*c)) + rho^2*z/(sigma*c^3)
    d2m[5,5] = N/(c^3) * (1 + 3*rho^2/(c^2))       # m_{rr}
    d2m[3,6] = d2m[6,3] = rho/(sigma^2*c)
    d2m[4,6] = d2m[6,4] = rho*z/(sigma^2*c)
    d2m[6,6] = 2*rho*v/(sigma^3*c)

    # Nonzero entries of m-Hessian
    d2u[3,6] = d2u[6,3] = 1/sigma^2
    d2u[4,6] = d2u[6,4] = z/sigma^2
    d2u[6,6] = 2*u/sigma^2

    # Collect Hessian
    e_6 = [0,0,0,0,0,1.0]
    H = gp*(dm*dm') + g*d2m - (du*du') - u*d2u + (1/sigma^2)*(e_6*e_6')

    return H
end

# Get mean Hessian
H_sum = zeros(6,6)
for i in 1:nrow(df)
    Hi = ivprobit_obs_llgh(theta_MLE_fast, df.y[i], df.x[i], df.z[i])
    H_sum .+= Hi
end

# Variance-covariance of parameters
V_hat = inv(-H_sum)
se_MLE = sqrt.(diag(V_hat))

hcat(theta_MLE_fast, se_MLE, theta_true)

6×3 Matrix{Float64}:
  0.429862  0.0839797   0.5
  0.993964  0.0887563   1.0
 -1.04197   0.0325769  -1.0
  1.94199   0.0573448   2.0
  0.585085  0.0517097   0.6
  1.02932   0.0230168   1.0

In [43]:
# Now let's give the GMM approach a shot. Note that we will have to define a weight matrix - for today, let's just use I

function gmm_objective(theta; data = df, W = I)
    z, x, y = data.z, data.x, data.y
    beta_c, beta_x, pi_c, pi_z = theta # note that theta is lower-dimensional - this is because we don't need to estimate the shock dist params!

    # Construct error terms
    error1 = x .- pi_c - pi_z .* z
    beta_xhat = beta_c .+ (beta_x .* (pi_c .+ pi_z .* z)) # beta times xhat
    error2 = y .- cdf.(Normal(), beta_xhat)
    
    # Construct moment vector
    moments = []
    moments = vcat(moments, [mean(error1)]) # error 1 interacted with the constant
    moments = vcat(moments, [mean(error1 .* z)]) # error 1 interacted with z
    moments = vcat(moments, [mean(error2)]) # error 2 interacted with the constant
    moments = vcat(moments, [mean(error2 .* z)]) # error 2 interacted with z
    return moments' * W * moments
end

# We need to define a new initial value, since we are estimating a different set of params
theta_init_gmm = theta_init[1:4]
theta_true_gmm = theta_true[1:4]

result_gmm = optimize(theta -> gmm_objective(theta; data = df, W = I),
                  theta_init_gmm,
                  NelderMead())

theta_gmm = Optim.minimizer(result_gmm)
hcat(theta_gmm, theta_true_gmm)

4×2 Matrix{Float64}:
  0.236089   0.5
  0.548748   1.0
 -1.04203   -1.0
  1.94206    2.0

In [44]:
# Our estimates for beta above are wrong, but they seem to have the right ratio - this is not a coincidence.
# The IV-probit model we are estimating is normalizing a slightly different error variance to 1. See the Wilde (2008) paper for details.
# I've written up code that renormalizes our estimates back to Var(e1) = 1. I haven't proven that this algorithm is consistent, but it seems to work.

function renormalization_param(theta; data = df) ### Andrew note-to-self: sometimes unstable. Debug more later.
    z, x, y = data.z, data.x, data.y
    beta_c, beta_x, pi_c, pi_z = theta

    error1 = x .- pi_c - pi_z .* z
    beta_xhat = beta_c .+ (beta_x .* (pi_c .+ pi_z .* z))
    error2 = y .- cdf.(Normal(), beta_xhat)
    
    error2_tilde = ifelse.(y .== 1, pdf.(Normal(), beta_xhat) / cdf.(Normal(), beta_xhat), pdf.(Normal(), beta_xhat) / (cdf.(Normal(), beta_xhat) .- 1))
    return sqrt(var(error2_tilde) + (beta_x^2) * var(error1) + 2 * beta_x * cov(error1, error2_tilde)[1])
end

sigma_hat = renormalization_param(theta_gmm, data = df)
theta_gmm_ren = vcat(theta_gmm[1:2] / sigma_hat, theta_gmm[3:4])
hcat(theta_gmm_ren, theta_true_gmm)

4×2 Matrix{Float64}:
  0.417446   0.5
  0.970282   1.0
 -1.04203   -1.0
  1.94206    2.0

In [45]:
# and we can get the standard errors too
# Really should inflate these a bit to deal with generated rescaling, but let's not worry about that for now.

function gmm_ses(theta; data = df, W = I)
    z, x, y = data.z, data.x, data.y
    beta_c, beta_x, pi_c, pi_z = theta 
    
    # Construct error terms
    error1 = x .- pi_c - pi_z .* z
    xhat = pi_c .+ pi_z .* z
    beta_xhat = beta_c .+ (beta_x .* xhat) # beta times xhat
    error2 = y .- cdf.(Normal(), beta_xhat)
    phi = pdf.(Normal(), beta_xhat)

    # Moment outer products
    Delta = zeros(4,4)
    for i in 1:N
        vec = [error1[i], error1[i] * z[i], error2[i], error2[i] * z[i]] # our 4 moment conditions 
        Delta .+=  vec * vec'
    end

    # Moment derivatives
    Gamma = zeros(4,4)
    for i in 1:N      
        #Gamma[1:2,1:2] = 0
        Gamma[1,3] += -1
        Gamma[1,4] += -z[i]
        
        Gamma[2,3] += -z[i]
        Gamma[2,4] += -(z[i]^2)
        
        Gamma[3,1] += -phi[i]
        Gamma[3,2] += -phi[i] * xhat[i]
        Gamma[3,3] += -phi[i] * beta_x
        Gamma[3,4] += -phi[i] * beta_x * z[i]
        
        Gamma[4,1] += -phi[i] * z[i]
        Gamma[4,2] += -phi[i] * xhat[i] * z[i]
        Gamma[4,3] += -phi[i] * beta_x * z[i]
        Gamma[4,4] += -phi[i] * beta_x * (z[i]^2)
    end
    
    V_hat = inv(Gamma' * W * Gamma) * Gamma' * W * Delta * W * Gamma * inv(Gamma' * W * Gamma)

    return Gamma, Delta, V_hat
end

_, Delta_step1, V_hat_gmm = gmm_ses(theta_gmm; data = df, W = I)
se_GMM = sqrt.(diag(V_hat_gmm))

# still need to renormalize
se_GMM_ren = vcat(se_GMM[1:2] / sigma_hat, se_GMM[3:4])

hcat(theta_gmm_ren, se_GMM_ren, theta_true_gmm)

4×3 Matrix{Float64}:
  0.417446  0.0780228   0.5
  0.970282  0.0583769   1.0
 -1.04203   0.0325905  -1.0
  1.94206   0.0570953   2.0

In [46]:
# and we can also run 2-step GMM to get a more efficient estimator

# rerun estimator with new weight matrix
result_gmm_2step = optimize(theta -> gmm_objective(theta; data = df, W = inv(Delta_step1)),
                  theta_init_gmm,
                  NelderMead())

theta_gmm_2step = Optim.minimizer(result_gmm_2step)

# get new SEs #given the new estimated theta 
_, _, V_hat_gmm_2step = gmm_ses(theta_gmm_2step; data = df, W = inv(Delta_step1))
se_GMM_2step = sqrt.(diag(V_hat_gmm_2step))

# still need to renormalize
sigma_hat_2step = renormalization_param(theta_gmm_2step, data = df)
theta_gmm_2step_ren = vcat(theta_gmm_2step[1:2] / sigma_hat_2step, theta_gmm_2step[3:4])
se_GMM_2step_ren = vcat(se_GMM_2step[1:2] / sigma_hat_2step, se_GMM_2step[3:4])

# basically no difference from the 1-step GMM in this case - likely because it's just-ID
hcat(theta_gmm_2step_ren, se_GMM_2step_ren, theta_true_gmm)

4×3 Matrix{Float64}:
  0.41485   0.0781449   0.5
  0.970278  0.0584733   1.0
 -1.04391   0.0325906  -1.0
  1.94019   0.0570979   2.0

In [47]:
# Let's replace the Normal CDFs above with a simulation equivalent to implement an MSL and an MSM estimator

# First we compute our random draws (which we will do exactly once!!!)
Random.seed!(73723462)
S = 100 # 500
U = rand(Normal(), S)
U_weight = ones(S) ./ S

# Then we write our Monte Carlo simulation of the integral 
# adding this clamp to avoid floating point imprecision that cause log(something < 0) problems
sim_Phi(x, U, U_weight) = clamp.((x .> U') * U_weight, 0.0, 1.0)

# Now we rewrite the objective function, but literally the only difference is that we replace cdf.(Normal(), xxx) with sim_Phi.(xxx, U, U_weight)

function msl_objective(theta; data = df, U = U, U_weight = U_weight)
    z, x, y = data.z, data.x, data.y
    beta_c, beta_x, pi_c, pi_z, rho, sigma = theta

    stage1 = pdf.(Normal(), (x .- pi_c .- pi_z .* z) ./ sigma) ./ sigma
    M = (beta_c .+ beta_x .* x .+ (rho / sigma) .* (x .- pi_c .- pi_z .* z)) ./ sqrt(1 - (rho^2))
    stage2 = ifelse.(y .== 1, sim_Phi(M, U, U_weight), 1 .- sim_Phi(M, U, U_weight))
    return -sum(log.(stage2 .* stage1))
end

result_msl = optimize(theta -> msl_objective(theta; data = df, U = U, U_weight = U_weight),
                  lower, upper,
                  theta_init,
                  Fminbox(NelderMead()))

# and the results are ok (better with higher S!). 
# Here, this code is a bit unstable - the optimizer sometimes is hitting an early stopping point. My guess is this is a Nelder-Mead issue.

theta_MSL = Optim.minimizer(result_msl)
hcat(theta_MSL, theta_MLE, theta_true)

6×3 Matrix{Float64}:
  0.4     0.429869   0.5
  1.1     0.993969   1.0
 -0.1    -1.04197   -1.0
  2.5     1.94199    2.0
  0.5     0.585083   0.6
  1.375   1.02932    1.0

In [48]:
# Another way to do it is to use a Quadrature rule - let's use the Gauss-Hermite quadrature from the Expectations.jl package

quad = expectation(Normal(), n = S)
U = nodes(quad)
U_weight = weights(quad)

result_msl_quad = optimize(theta -> msl_objective(theta; data = df, U = U, U_weight = U_weight),
                  lower, upper,
                  theta_init,
                  Fminbox(NelderMead()))

# and these are much much much better!

theta_MSL_quad = Optim.minimizer(result_msl_quad)
hcat(theta_MSL_quad, theta_MSL, theta_MLE, theta_true)

6×4 Matrix{Float64}:
  0.465794   0.4     0.429869   0.5
  1.01793    1.1     0.993969   1.0
 -1.04197   -0.1    -1.04197   -1.0
  1.94199    2.5     1.94199    2.0
  0.607151   0.5     0.585083   0.6
  1.02932    1.375   1.02932    1.0

In [49]:
# and for the sake of completeness let's run a MSM using that quadrature rule

function msm_objective(theta; data = df, W = I)
    z, x, y = data.z, data.x, data.y
    beta_c, beta_x, pi_c, pi_z = theta # note that theta is lower-dimensional - this is because we don't need to estimate the shock dist params!

    # Construct error terms
    error1 = x .- pi_c - pi_z .* z
    beta_xhat = beta_c .+ (beta_x .* (pi_c .+ pi_z .* z)) # beta times xhat
    error2 = y .- sim_Phi(beta_xhat, U, U_weight)
    
    # Construct moment vector
    moments = []
    moments = vcat(moments, [mean(error1)]) # error 1 interacted with the constant
    moments = vcat(moments, [mean(error2)]) # error 2 interacted with the constant
    moments = vcat(moments, [mean(error1 .* z)]) # error 1 interacted with z
    moments = vcat(moments, [mean(error2 .* z)]) # error 2 interacted with z
    return moments' * W * moments
end

result_msm = optimize(theta -> msm_objective(theta; data = df, W = I),
                  theta_init_gmm,
                  NelderMead())

theta_msm = Optim.minimizer(result_msm)

#renormalization - not bothering with adding the simulation Phi() within this function, but could do that too
sigma_hat = renormalization_param(theta_msm, data = df)
theta_msm_ren = vcat(theta_msm[1:2] / sigma_hat, theta_msm[3:4])

#and the results look pretty much identical again

hcat(theta_msm_ren, theta_gmm_2step_ren, theta_true_gmm)

4×3 Matrix{Float64}:
  0.466125   0.41485    0.5
  0.97103    0.970278   1.0
 -1.04196   -1.04391   -1.0
  1.94201    1.94019    2.0